In [9]:
!pip install openai azure-core


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import os
import asyncio
import json
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional
from datetime import datetime
from azure.core.credentials import AzureKeyCredential
from openai import AsyncAzureOpenAI



# 2. Initialize the Azure OpenAI Client
async_client = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)

print(" Azure OpenAI Client Initialized Successfully!")
print(f"   Endpoint: {AZURE_OPENAI_ENDPOINT}")
print(f"   Deployment Target: {AZURE_OPENAI_DEPLOYMENT}")

 Azure OpenAI Client Initialized Successfully!
   Endpoint: https://azure-foundry-08.openai.azure.com
   Deployment Target: gpt-5


In [11]:
@dataclass
class SettlementOrderRecord:
    order_id: str
    sku: str
    amount: float
    fba_fee: float
    commission: float
    promo_discount: float
    expected_net: float
    calculated_net: float = 0.0
    status: str = "UNPROCESSED"

@dataclass
class AsyncReconciliationState:
    settlement_id: str
    raw_payload: str
    orders: List[SettlementOrderRecord] = field(default_factory=list)
    current_node: str = "START"
    retry_count: int = 0
    max_retries: int = 3
    loop_exited: bool = False
    is_completed: bool = False
    audit_trail: List[str] = field(default_factory=list)

    def log(self, node_name: str, message: str):
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S.%f")[:-3]
        entry = f"[{timestamp}] [{node_name}] {message}"
        self.audit_trail.append(entry)
        print(entry)

# Test State Initialization
sample_state = AsyncReconciliationState(
    settlement_id="SET-AZN-2026-BATCH10",
    raw_payload="MOCK_RAW_PAYLOAD_10_ORDERS"
)
sample_state.log("INIT", "Initialized AsyncReconciliationState for 10-order settlement batch.")

[2026-08-18 04:37:53.993] [INIT] Initialized AsyncReconciliationState for 10-order settlement batch.


In [12]:
# Simulated Dynamics 365 ERP Ledger for 10 Amazon Orders
MOCK_D365_ERP_BATCH = {
    f"112-2026-{i:02d}": {"expected_net": round(100.0 + (i * 15.0) - (5.0 if i % 3 == 0 else 0.0), 2)}
    for i in range(1, 11)
}

async def async_ingestion_node(state: AsyncReconciliationState) -> str:
    state.current_node = "INGESTION_NODE"
    state.log(state.current_node, f"Ingesting raw settlement batch: {state.settlement_id}...")
    await asyncio.sleep(0.1) # Simulate async I/O payload validation
    
    # Generate 10 Sample Amazon Order Records
    orders = []
    for i in range(1, 11):
        order_id = f"112-2026-{i:02d}"
        amount = round(150.0 + (i * 20.0), 2)
        fba_fee = -15.0
        commission = -22.5
        # Inject an unallocated promo discount anomaly on order 5
        promo_discount = -550.00 if i == 5 else (-5.0 if i % 3 == 0 else 0.0)
        expected = MOCK_D365_ERP_BATCH[order_id]["expected_net"]
        
        orders.append(SettlementOrderRecord(
            order_id=order_id,
            sku=f"NW-ITEM-{i:02d}",
            amount=amount,
            fba_fee=fba_fee,
            commission=commission,
            promo_discount=promo_discount,
            expected_net=expected
        ))
    
    state.orders = orders
    state.log(state.current_node, f"Successfully ingested and format-validated {len(orders)} order records.")
    return "EXTRACTION_NODE"

async def async_extraction_node(state: AsyncReconciliationState) -> str:
    state.current_node = "EXTRACTION_NODE"
    state.log(state.current_node, "Extracting line items and calculating net payouts...")
    await asyncio.sleep(0.1)
    
    for order in state.orders:
        order.calculated_net = round(
            order.amount + order.fba_fee + order.commission + order.promo_discount, 2
        )
        order.status = "EXTRACTED"
        
    state.log(state.current_node, f"Extracted remittance fields across {len(state.orders)} orders.")
    return "MATCHING_NODE"

async def async_matching_node(state: AsyncReconciliationState) -> str:
    state.current_node = "MATCHING_NODE"
    state.log(state.current_node, "Performing 3-way matching against Dynamics 365 ERP ledger...")
    await asyncio.sleep(0.1)
    
    unresolved_variances = 0
    for order in state.orders:
        variance = round(abs(order.expected_net - order.calculated_net), 2)
        if variance == 0.0:
            order.status = "MATCHED_PERFECT"
        elif abs(order.promo_discount) > 500.0:
            order.status = "UNALLOCATED_PROMO_VARIANCE"
            unresolved_variances += 1
        else:
            order.status = "MATCHED_WITH_MINOR_VARIANCE"

    state.log(state.current_node, f"Matching complete. Perfect: {len(state.orders)-unresolved_variances}, Unresolved Variances: {unresolved_variances}")
    
    if unresolved_variances > 0:
        return "EVALUATE_RETRY_LOOP"
    return "COMPLETE"

In [13]:
async def async_evaluate_retry_loop_node(state: AsyncReconciliationState) -> str:
    state.current_node = "EVALUATE_RETRY_LOOP"
    state.retry_count += 1
    
    state.log(
        state.current_node, 
        f"Evaluating unresolved variances. Retry Attempt {state.retry_count}/{state.max_retries}..."
    )
    await asyncio.sleep(0.05)
    
    # Check Loop-Exit Condition
    if state.retry_count >= state.max_retries:
        state.loop_exited = True
        state.log(
            state.current_node, 
            f" LOOP-EXIT CONDITION TRIPPED! Max retries ({state.max_retries}) reached for unallocated promo variances. Breaking loop to prevent infinite execution cycle."
        )
        return "COMPLETE"
    else:
        state.log(state.current_node, "Re-routing back to MATCHING_NODE for smart variance reallocation...")
        return "MATCHING_NODE"

In [14]:
import asyncio

class AsyncGraphOrchestrator:
    def __init__(self, state: AsyncReconciliationState):
        self.state = state
        self._nodes = {
            "INGESTION_NODE": async_ingestion_node,
            "EXTRACTION_NODE": async_extraction_node,
            "MATCHING_NODE": async_matching_node,
            "EVALUATE_RETRY_LOOP": async_evaluate_retry_loop_node,
        }

    async def run(self):
        self.state.log("ORCHESTRATOR", f"Starting state graph processing for Settlement {self.state.settlement_id}...")
        
        next_node = "INGESTION_NODE"
        while next_node != "COMPLETE":
            node_handler = self._nodes.get(next_node)
            if node_handler:
                next_node = await node_handler(self.state)
            else:
                self.state.log("ORCHESTRATOR", f" Unknown node '{next_node}'. Halting execution.")
                return

        self.state.is_completed = True
        self.state.log("ORCHESTRATOR", "State graph execution completed successfully.")

# Execution based on environment:

# Option A: Interactive / Notebook / Active Event Loop
execution_state = AsyncReconciliationState(
    settlement_id="SET-AZN-2026-10ORDERS",
    raw_payload="RAW_CSV_INPUT_10_ORDERS"
)
orchestrator = AsyncGraphOrchestrator(execution_state)
await orchestrator.run()

# Option B: Standard Python Script Entrypoint
# if __name__ == "__main__":
#     asyncio.run(orchestrator.run())

[2026-08-18 04:37:54.042] [ORCHESTRATOR] Starting state graph processing for Settlement SET-AZN-2026-10ORDERS...
[2026-08-18 04:37:54.043] [INGESTION_NODE] Ingesting raw settlement batch: SET-AZN-2026-10ORDERS...


[2026-08-18 04:37:54.143] [INGESTION_NODE] Successfully ingested and format-validated 10 order records.
[2026-08-18 04:37:54.143] [EXTRACTION_NODE] Extracting line items and calculating net payouts...
[2026-08-18 04:37:54.243] [EXTRACTION_NODE] Extracted remittance fields across 10 orders.
[2026-08-18 04:37:54.243] [MATCHING_NODE] Performing 3-way matching against Dynamics 365 ERP ledger...
[2026-08-18 04:37:54.344] [MATCHING_NODE] Matching complete. Perfect: 9, Unresolved Variances: 1
[2026-08-18 04:37:54.344] [EVALUATE_RETRY_LOOP] Evaluating unresolved variances. Retry Attempt 1/3...
[2026-08-18 04:37:54.394] [EVALUATE_RETRY_LOOP] Re-routing back to MATCHING_NODE for smart variance reallocation...
[2026-08-18 04:37:54.394] [MATCHING_NODE] Performing 3-way matching against Dynamics 365 ERP ledger...
[2026-08-18 04:37:54.494] [MATCHING_NODE] Matching complete. Perfect: 9, Unresolved Variances: 1
[2026-08-18 04:37:54.495] [EVALUATE_RETRY_LOOP] Evaluating unresolved variances. Retry Atte

In [15]:
print("==========================================================================================================")
print(f"       END-TO-END ASYNC ORCHESTRATION REPORT: SETTLEMENT {execution_state.settlement_id}")
print("==========================================================================================================")
print(f"📊 Final State Completion: {execution_state.is_completed}")
print(f"🔄 Total Retries Attempted: {execution_state.retry_count}/{execution_state.max_retries}")
print(f"🚨 Loop Exit Tripped: {execution_state.loop_exited}")
print(f"📦 Total Orders Processed: {len(execution_state.orders)}")

print("\n----------------------------------------------------------------------------------------------------------")
print(f"{'ORDER ID':<15} | {'SKU':<12} | {'CALC NET ($)':<12} | {'EXP NET ($)':<12} | {'VARIANCE ($)':<12} | {'STATUS':<25}")
print("----------------------------------------------------------------------------------------------------------")

for o in execution_state.orders:
    var = round(abs(o.expected_net - o.calculated_net), 2)
    print(f"{o.order_id:<15} | {o.sku:<12} | ${o.calculated_net:<11.2f} | ${o.expected_net:<11.2f} | ${var:<11.2f} | {o.status:<25}")

print("----------------------------------------------------------------------------------------------------------\n")
print("--- CHRONOLOGICAL GRAPH AUDIT LOG ---")
for entry in execution_state.audit_trail:
    print(entry)
print("==========================================================================================================")

       END-TO-END ASYNC ORCHESTRATION REPORT: SETTLEMENT SET-AZN-2026-10ORDERS
📊 Final State Completion: True
🔄 Total Retries Attempted: 3/3
🚨 Loop Exit Tripped: True
📦 Total Orders Processed: 10

----------------------------------------------------------------------------------------------------------
ORDER ID        | SKU          | CALC NET ($) | EXP NET ($)  | VARIANCE ($) | STATUS                   
----------------------------------------------------------------------------------------------------------
112-2026-01     | NW-ITEM-01   | $132.50      | $115.00      | $17.50       | MATCHED_WITH_MINOR_VARIANCE
112-2026-02     | NW-ITEM-02   | $152.50      | $130.00      | $22.50       | MATCHED_WITH_MINOR_VARIANCE
112-2026-03     | NW-ITEM-03   | $167.50      | $140.00      | $27.50       | MATCHED_WITH_MINOR_VARIANCE
112-2026-04     | NW-ITEM-04   | $192.50      | $160.00      | $32.50       | MATCHED_WITH_MINOR_VARIANCE
112-2026-05     | NW-ITEM-05   | $-337.50     | $175.00      